# Baseline modele: LSTM i Transformer

Notebook definiuje pipeline do porównania baseline modeli dla rozpoznawania krótkich komend głosowych. Nie uruchamia eksperymentów automatycznie; konfiguracje, modele, dataset, trening i zapis wyników znajdują się w modułach `scripts`.

## Konfiguracja

Importy z lokalnego pakietu oraz podstawowe etykiety zadania.

In [ ]:
from scripts import (
    DataFixedParams,
    DataGridParams,
    Experiment,
    FeatureFixedParams,
    FitFixedParams,
    FitGridParams,
    LABEL_ORDER,
    ModelGridParams,
    experiment_grid_dataframe,
    run_experiment,
)

LABEL_ORDER

## Interfejs eksperymentów

Eksperyment jest opisany przez dataclassy. Parametry stałe opisują dane, cechy i trening, a klasy `GridParams` pozwalają podać pojedynczą wartość albo listę wartości do porównania.

In [ ]:
default_experiment = Experiment(name="default_full_data_baseline")
experiment_grid_dataframe(default_experiment)

## Pipeline danych

Moduł `scripts.data` buduje manifest z `train.7z`, mapuje klasy spoza listy docelowej do `unknown`, tworzy przykłady `silence` z `_background_noise_` i przygotowuje mel-spektrogramy przez `torchaudio`. Oficjalny `testing_list.txt` jest używany jako test z etykietami; Kaggle `test.7z` nie służy do metryk, bo nie zawiera etykiet.

In [ ]:
data_fixed = DataFixedParams()
feature_fixed = FeatureFixedParams()

data_fixed, feature_fixed

## Modele baseline

Moduł `scripts.models` zawiera `LSTMBaseline` i `TransformerBaseline`. Oba modele przyjmują tę samą reprezentację wejściową: sekwencję ramek mel-spektrogramu w formacie `[czas, pasma_mel]`.

In [ ]:
model_grid = ModelGridParams(
    model_type=["lstm", "transformer"],
    dropout=0.2,
    lstm_hidden_size=128,
    lstm_layers=2,
    lstm_bidirectional=True,
    transformer_d_model=128,
    transformer_heads=4,
    transformer_layers=2,
    transformer_ff_dim=256,
)

model_grid

## Wyniki i wykresy

Po uruchomieniu `run_experiment` każdy run zapisuje konfigurację, manifest, metryki, krzywe uczenia i macierz pomyłek do katalogu `reports/02_baseline_models/<experiment>/<run>/`.

In [ ]:
fit_fixed = FitFixedParams(device="cpu", use_tqdm=True)
fit_grid = FitGridParams(
    epochs=5,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
)

fit_fixed, fit_grid

## Definicja baseline eksperymentu

Domyślne klasy konfiguracyjne używają pełnych danych. Poniższy eksperyment startowy jest ustawiony na 5% splitu treningowego i 5% splitu testowego, żeby można było szybko sprawdzić pipeline po odkomentowaniu uruchomienia.

In [ ]:
baseline_experiment = Experiment(
    name="baseline_lstm_transformer_5_percent",
    data_fixed=data_fixed,
    feature_fixed=feature_fixed,
    fit_fixed=fit_fixed,
    data_grid=DataGridParams(
        train_fraction=0.05,
        test_fraction=0.05,
        unknown_fraction=0.05,
        silence_examples_per_split=100,
        sampling_strategy="natural",
        seed=42,
    ),
    model_grid=model_grid,
    fit_grid=fit_grid,
)

experiment_grid_dataframe(baseline_experiment)

## Uruchomienie eksperymentu

Poniższa komórka jest celowo zakomentowana. Jej odkomentowanie uruchomi oba baseline modele i zapisze wyniki w strukturze katalogów opisanej wyżej.

In [ ]:
# results = run_experiment(baseline_experiment)
# results